# 3.6 Mixed Feature DNN

這份 Notebook 示範數值欄位與類別欄位混合的表格資料模型。

## 1. 載入套件與建立資料

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

tf.keras.utils.set_random_seed(42)
rng = np.random.default_rng(42)

n = 2500
df = pd.DataFrame({
    'age': rng.normal(36, 10, n).clip(18, 75),
    'income': rng.lognormal(mean=10.4, sigma=0.35, size=n),
    'visits': rng.poisson(5, n),
    'city': rng.choice(['taipei', 'taichung', 'tainan', 'kaohsiung'], n, p=[0.35, 0.25, 0.2, 0.2]),
    'plan': rng.choice(['free', 'basic', 'pro'], n, p=[0.45, 0.35, 0.2]),
    'device': rng.choice(['mobile', 'desktop', 'tablet'], n, p=[0.55, 0.35, 0.1]),
})
logit = (
    0.04 * (df['age'] - 36)
    + 0.00003 * (df['income'] - df['income'].mean())
    + 0.18 * (df['visits'] - 5)
    + (df['city'] == 'taipei') * 0.5
    + (df['plan'] == 'pro') * 1.0
    + (df['device'] == 'desktop') * 0.35
)
df['target'] = (logit + rng.normal(0, 0.9, n) > 0.8).astype('int64')
df.head()

## 2. 定義欄位並切分資料

In [ ]:
numeric_cols = ['age', 'income', 'visits']
categorical_cols = ['city', 'plan', 'device']
target_col = 'target'

train_df, test_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df[target_col])
print('train:', train_df.shape)
print('test:', test_df.shape)
print('positive ratio:', train_df[target_col].mean().round(3), test_df[target_col].mean().round(3))

## 3. 建立數值與類別前處理 layer

所有 preprocessing layer 只使用訓練資料 adapt。

In [ ]:
normalizer = tf.keras.layers.Normalization(name='numeric_normalization')
normalizer.adapt(train_df[numeric_cols].astype('float32').values)

lookup_layers = {}
for col in categorical_cols:
    lookup = tf.keras.layers.StringLookup(output_mode='one_hot', name=f'{col}_lookup')
    lookup.adapt(tf.constant(train_df[col].astype(str).values.reshape(-1, 1)))
    lookup_layers[col] = lookup
    print(col, lookup.get_vocabulary())

## 4. 建立 Functional API 模型

In [ ]:
numeric_input = tf.keras.Input(shape=(len(numeric_cols),), dtype=tf.float32, name='numeric')
encoded_features = [normalizer(numeric_input)]
inputs = {'numeric': numeric_input}

for col in categorical_cols:
    inp = tf.keras.Input(shape=(1,), dtype=tf.string, name=col)
    inputs[col] = inp
    encoded_features.append(lookup_layers[col](inp))

features = tf.keras.layers.Concatenate()(encoded_features)
x = tf.keras.layers.Dense(32, activation='relu')(features)
x = tf.keras.layers.Dense(16, activation='relu')(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=inputs, outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
model.summary()

## 5. 整理 Keras 輸入格式

In [ ]:
def make_inputs(frame):
    data = {'numeric': frame[numeric_cols].astype('float32').values}
    for col in categorical_cols:
        data[col] = tf.constant(frame[col].astype(str).values.reshape(-1, 1))
    return data

train_inputs = make_inputs(train_df)
test_inputs = make_inputs(test_df)
y_train = train_df[target_col].astype('float32').values
y_test = test_df[target_col].astype('float32').values

## 6. 訓練與評估

In [ ]:
history = model.fit(
    train_inputs, y_train,
    validation_split=0.2,
    epochs=40,
    batch_size=32,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)]
)

pd.DataFrame([
    model.evaluate(train_inputs, y_train, verbose=0, return_dict=True),
    model.evaluate(test_inputs, y_test, verbose=0, return_dict=True),
], index=['train', 'test'])

## 7. 預測與分類報告

In [ ]:
prob = model.predict(test_inputs, verbose=0).ravel()
y_pred = (prob >= 0.5).astype(int)
print(classification_report(y_test, y_pred, digits=4))

preview = test_df[numeric_cols + categorical_cols + [target_col]].head().copy()
preview['probability'] = prob[:5]
preview['predicted'] = y_pred[:5]
preview

## 8. 小結

混合欄位 DNN 使用 Functional API 較清楚：數值欄位做 Normalization，類別欄位做 StringLookup，再 Concatenate 後進入 Dense layers。